<a href="https://colab.research.google.com/github/Mikop22/AttentionIsAllYouNeed/blob/main/SENTIMENT%20ANALYSIS%20NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Preprocessing

In [1]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

In [2]:
# External hook: lightweight language detection. Replace with project detector if available.
try:
    from .lang_detect import detect_language  # type: ignore
except Exception:
    # Fallback stub if import fails; returns 'unknown'
    def detect_language(text: str) -> str:
        return "unknown"

Output formats

In [13]:
from dataclasses import dataclass, field

@dataclass
class Metadata:
    """
    Metadata describing the text input and processing outcomes.

    Fields:
    - source: Optional identifier for the source platform (e.g., 'twitter', 'reddit').
    - language: ISO language code detected or provided (e.g., 'en'). 'unknown' if detection fails.
    - timestamp: Optional raw ingest timestamp as string or int (no transformation performed).
    - discarded: True if the input is considered non-processable (empty or only noise).
    - notes: Arbitrary processing notes (e.g., 'lang_unsupported', 'url_only').
    """
    source: Optional[str] = None
    language: str = "unknown"
    timestamp: Optional[Any] = None
    discarded: bool = False
    notes: List[str] = field(default_factory=list)

In [20]:
from dataclasses import dataclass, field

@dataclass
class PreprocessorResult:
    """
    Result structure holding cleaned text and updated metadata.

    Fields:
    - text: Cleaned text string or None if discarded.
    - metadata: Updated Metadata instance with detection outcomes and flags.
    """
    text: Optional[str]
    metadata: Metadata

In [18]:
from dataclasses import dataclass, field

@dataclass
class PreprocessorConfig:
    """
    Configurable options for the text preprocessing pipeline.

    Fields:
    - lower_case: Convert text to lowercase. Some languages may prefer preserving case.
    - emoji_policy: 'remove' to strip emojis, 'map' to convert to sentiment tokens.
    - remove_handles: Remove social media handles like '@username'.
    - remove_urls: Remove hyperlinks (http/https/www and bare domains).
    - remove_html_markdown: Strip HTML tags and basic markdown formatting.
    - normalize_punctuation: Collapse repeated punctuation (e.g., '!!!' -> '!').
    - preserve_year_like_numbers: Keep 4-digit numbers likely to be years; remove other excessive numbers if noisy.
    - remove_stopwords: Remove common stopwords while preserving negations.
    - supported_languages: List of ISO codes allowed; if empty, all languages are allowed.
    - route_unsupported_to_multilingual: If True, do not discard unsupported languages; leave metadata note.
    - min_content_chars: Minimum number of alphabetic characters required to keep the text (post-cleaning).
    """
    lower_case: bool = True
    emoji_policy: str = "map"  # 'map' or 'remove'
    remove_handles: bool = True
    remove_urls: bool = True
    remove_html_markdown: bool = True
    normalize_punctuation: bool = True
    preserve_year_like_numbers: bool = True
    remove_stopwords: bool = False
    supported_languages: List[str] = field(default_factory=list)  # e.g., ['en', 'fr']
    route_unsupported_to_multilingual: bool = True
    min_content_chars: int = 2

REGEX:

In [9]:
URL_RE = re.compile(
    r"""(?xi)
    \b
    (
        (https?://|www\.)[^\s]+      # http(s) or www
        |
        [a-z0-9.-]+\.[a-z]{2,}(/[^\s]*)?  # bare domain like example.com/path
    )
    """
)

HANDLE_RE = re.compile(r"(^|[\s\W])@[\w_]{1,50}", re.UNICODE)

HTML_TAG_RE = re.compile(r"<[^>]+>")
MD_FORMAT_RE = re.compile(r"(?m)(`{1,3}.*?`{1,3}|\*{1,2}|_{1,2}|~~)")

PUNCT_REPEAT_RE = re.compile(r"([!?.,;:])\1+")
WHITESPACE_RE = re.compile(r"\s+")

# Unicode emoji ranges (basic approximation); can be extended.
EMOJI_RE = re.compile(
    r"[\U0001F300-\U0001F5FF"  # Misc Symbols and Pictographs
    r"\U0001F600-\U0001F64F"   # Emoticons
    r"\U0001F680-\U0001F6FF"   # Transport & Map Symbols
    r"\U0001F700-\U0001F77F"   # Alchemical Symbols
    r"\U0001F780-\U0001F7FF"   # Geometric Shapes Extended
    r"\U0001F800-\U0001F8FF"   # Supplemental Arrows-C
    r"\U0001F900-\U0001F9FF"   # Supplemental Symbols and Pictographs
    r"\U0001FA00-\U0001FA6F"   # Symbols and Pictographs Extended-A
    r"\U00002700-\U000027BF"   # Dingbats
    r"\U00002600-\U000026FF]"  # Misc Symbols
, flags=re.UNICODE)

# Accept digits; we may remove non-year-like numbers.
NON_YEAR_NUMBER_RE = re.compile(r"\b(?!\d{4}\b)\d+\b")


# --- Emoji sentiment map (basic) ---
EMOJI_SENTIMENT_MAP = {
    "🙂": "smile",
    "😊": "smile",
    "😃": "smile",
    "😀": "smile",
    "🤣": "laugh",
    "😂": "laugh",
    "😉": "wink",
    "😍": "love",
    "😘": "kiss",
    "😡": "angry",
    "🤬": "angry",
    "😠": "angry",
    "😢": "sad",
    "😭": "cry",
    "😞": "sad",
    "😔": "sad",
    "😐": "neutral",
    "😕": "confused",
    "😬": "grimace",
    "😱": "shock",
    "😮": "surprised",
    "👍": "thumbs_up",
    "👎": "thumbs_down",
    "🔥": "fire",
    "💯": "hundred",
    "✨": "sparkle",
    "❤️": "heart",
    "💔": "broken_heart",
}


# --- Stopword sets with preserved negations ---
EN_STOPWORDS = {
    "the", "a", "an", "and", "or", "if", "then", "but", "so", "to", "of", "in",
    "on", "for", "with", "from", "by", "as", "at", "that", "this", "it", "is",
    "are", "was", "were", "be", "been", "being", "do", "does", "did", "can",
    "could", "should", "would", "may", "might", "will", "shall", "have", "has",
    "had", "about", "into", "over", "under", "up", "down", "out", "off"
    # Note: Negations intentionally excluded: 'not', 'never', 'no'
}

FR_STOPWORDS = {
    "le", "la", "les", "un", "une", "des", "et", "ou", "si", "alors", "mais",
    "donc", "de", "du", "dans", "sur", "pour", "avec", "par", "comme", "à",
    "que", "ce", "cela", "il", "elle", "est", "sont", "était", "étaient",
    "être", "été", "ayant", "fait", "faites", "peut", "pourrait", "devrait",
    "voudrait", "peut-être", "avoir", "a", "ont"
    # Negations excluded: 'ne', 'pas', 'jamais', 'rien'
}

Functions to strip text:

In [15]:
def _strip_whitespace_and_case(text: str, lower_case: bool) -> str:
    """
    Normalize whitespace and case.

    Inputs:
    - text: Raw input text.
    - lower_case: If True, convert to lowercase.

    Output:
    - Normalized text with trimmed boundaries and optional lowercase.
    """
    text = text.strip()
    if lower_case:
        text = text.lower()
    # Replace various whitespace forms with a single space
    text = WHITESPACE_RE.sub(" ", text)
    return text


def _remove_urls(text: str) -> str:
    """
    Remove hyperlinks and bare domains.

    Inputs:
    - text: Current text.

    Output:
    - Text with URLs/domains removed.
    """
    return URL_RE.sub(" ", text)


def _remove_handles(text: str) -> str:
    """
    Remove social media handles (@username).

    Inputs:
    - text: Current text.

    Output:
    - Text with handles removed.
    """
    # Replace the handle with a single space while preserving preceding boundary
    return re.sub(r"(^|[\s\W])@[\w_]{1,50}", r"\1 ", text)


def _normalize_punctuation(text: str) -> str:
    """
    Collapse repeated punctuation and normalize spacing around punctuation.

    Inputs:
    - text: Current text.

    Output:
    - Text with normalized punctuation.
    """
    text = PUNCT_REPEAT_RE.sub(r"\1", text)
    # Remove stray non-word symbols except basic punctuation
    text = re.sub(r"[^\w\s.!?,;:'\"-]", " ", text)
    return text


def _handle_emojis(text: str, policy: str) -> Tuple[str, bool]:
    """
    Handle emojis per policy.

    Inputs:
    - text: Current text.
    - policy: 'remove' or 'map'.

    Output:
    - (Processed text, had_emoji)
    """
    had_emoji = False
    if policy == "remove":
        if EMOJI_RE.search(text):
            had_emoji = True
        text = EMOJI_RE.sub(" ", text)
        return text, had_emoji
    elif policy == "map":
        # Replace known emojis with sentiment tokens, strip unknowns
        def repl(match: re.Match) -> str:
            nonlocal had_emoji
            ch = match.group(0)
            had_emoji = True
            token = EMOJI_SENTIMENT_MAP.get(ch)
            return f" {token} " if token else " "

        text = EMOJI_RE.sub(repl, text)
        return text, had_emoji
    else:
        # Unknown policy; leave text unchanged
        return text, had_emoji


def _remove_noisy_numbers(text: str, preserve_year_like: bool) -> str:
    """
    Remove numbers considered noisy while preserving likely years (4-digit).

    Inputs:
    - text: Current text.
    - preserve_year_like: If True, preserve 4-digit numbers.

    Output:
    - Text with noisy numbers removed.
    """
    if preserve_year_like:
        text = NON_YEAR_NUMBER_RE.sub(" ", text)
    else:
        text = re.sub(r"\b\d+\b", " ", text)
    return text


def _remove_stopwords(text: str, language: str) -> str:
    """
    Optionally remove stopwords based on detected language, preserving negations.

    Inputs:
    - text: Current text.
    - language: ISO language code (e.g., 'en', 'fr').

    Output:
    - Text with stopwords removed for English or French; noop for others.
    """
    if language == "en":
        stopwords = EN_STOPWORDS
    elif language == "fr":
        stopwords = FR_STOPWORDS
    else:
        return text

    # Token-level removal with whitespace preservation
    tokens = text.split()
    filtered = [t for t in tokens if t not in stopwords]
    return " ".join(filtered)


def _final_spacing(text: str) -> str:
    """
    Ensure clean spacing and remove artifacts post-processing.

    Inputs:
    - text: Current text.

    Output:
    - Sanitized text with single spaces and trimmed ends.
    """
    text = WHITESPACE_RE.sub(" ", text)
    text = text.strip()
    return text


def _is_only_noise(text: str, min_content_chars: int) -> bool:
    """
    Determine if text is effectively noise (too few alphabetic characters).

    Inputs:
    - text: Current text.
    - min_content_chars: Minimum count of alphabetic characters required.

    Output:
    - True if considered noise, else False.
    """
    # Count alphabetic characters (any Unicode letter)
    alpha_chars = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]", text)
    return len(alpha_chars) < min_content_chars

**ACTION**

In [10]:
def preprocess(
    text: Optional[str],
    metadata: Optional[Metadata] = None,
    config: Optional[PreprocessorConfig] = None,
) -> PreprocessorResult:
    """
    Preprocess a single raw text input.

    Inputs:
    - text: Raw text string (may be None or empty).
    - metadata: Optional Metadata object; if None, a new one is created.
    - config: Optional PreprocessorConfig; defaults applied if None.

    Outputs:
    - PreprocessorResult with cleaned text (or None if discarded) and updated metadata.

    Logic flow:
    1. Validate input; if empty, mark discarded.
    2. Normalize whitespace and case.
    3. Remove URLs and handles per config.
    4. Strip HTML and markdown artifacts if enabled.
    5. Handle emojis (remove or map).
    6. Normalize punctuation and remove noisy numbers.
    7. Detect language; decide stopword handling.
    8. Remove stopwords if enabled; keep negations.
    9. Final spacing and noise check.
    10. Route unsupported languages per configuration.
    """
    md = metadata or Metadata()
    cfg = config or PreprocessorConfig()

    if text is None:
        md.discarded = True
        md.notes.append("empty_input")
        return PreprocessorResult(text=None, metadata=md)

    # Step 1: Early trim and case
    current = _strip_whitespace_and_case(text, cfg.lower_case)

    # If after trimming we have nothing, discard
    if not current:
        md.discarded = True
        md.notes.append("empty_after_trim")
        return PreprocessorResult(text=None, metadata=md)

    # Step 2: Remove URLs and handles
    if cfg.remove_urls:
        before = current
        current = _remove_urls(current)
        if before != current:
            md.notes.append("urls_removed")

    if cfg.remove_handles:
        before = current
        current = _remove_handles(current)
        if before != current:
            md.notes.append("handles_removed")


    # Step 4: Emoji handling
    current, had_emoji = _handle_emojis(current, cfg.emoji_policy)
    if had_emoji:
        md.notes.append(f"emoji_{cfg.emoji_policy}")

    # Step 5: Punctuation normalization
    if cfg.normalize_punctuation:
        before = current
        current = _normalize_punctuation(current)
        if before != current:
            md.notes.append("punct_normalized")

    # Step 6: Number handling
    before = current
    current = _remove_noisy_numbers(current, cfg.preserve_year_like_numbers)
    if before != current:
        md.notes.append("numbers_processed")

    # Step 7: Language detection
    try:
        detected = detect_language(current) if current else "unknown"
    except Exception:
        detected = "unknown"
        md.notes.append("lang_detect_error")
    md.language = detected or "unknown"

    # Step 8: Stopword handling (optional)
    if cfg.remove_stopwords:
        before = current
        current = _remove_stopwords(current, md.language)
        if before != current:
            md.notes.append("stopwords_removed")

    # Step 9: Final spacing and noise check
    current = _final_spacing(current)
    if not current or _is_only_noise(current, cfg.min_content_chars):
        md.discarded = True
        md.notes.append("only_noise")
        return PreprocessorResult(text=None, metadata=md)

    # Step 10: Language routing / support check
    if cfg.supported_languages and md.language not in cfg.supported_languages:
        if cfg.route_unsupported_to_multilingual:
            md.notes.append("lang_unsupported_routed")
        else:
            md.discarded = True
            md.notes.append("lang_unsupported_discarded")
            return PreprocessorResult(text=None, metadata=md)

    return PreprocessorResult(text=current, metadata=md)

In [11]:
def preprocess_batch(
    texts: List[Optional[str]],
    metadatas: Optional[List[Optional[Metadata]]] = None,
    config: Optional[PreprocessorConfig] = None,
) -> List[PreprocessorResult]:
    """
    Preprocess a batch of texts efficiently.

    Inputs:
    - texts: List of raw text strings (items may be None).
    - metadatas: Optional list of Metadata; aligns by index. Missing entries create default Metadata.
    - config: Optional PreprocessorConfig applied to all items.

    Outputs:
    - List of PreprocessorResult in the same order as inputs.

    Notes:
    - Designed for large batches; processing steps are lightweight and avoid per-item heavy allocations.
    """
    cfg = config or PreprocessorConfig()
    results: List[PreprocessorResult] = []

    md_list = metadatas or [None] * len(texts)
    if len(md_list) != len(texts):
        raise ValueError("Length of metadatas must match length of texts.")

    for i, raw in enumerate(texts):
        md = md_list[i] if isinstance(md_list[i], Metadata) else None
        results.append(preprocess(raw, md, cfg))

    return results

test

In [23]:

raw_text = """OMG!!! 😡😡
Check this out: link
& share with @user123"""

metadata = Metadata(
    source="twitter",
    language="unknown",   # will be updated by lang_detect
    timestamp="2025-11-18T13:06:00Z",
    discarded=False,
    notes=[]
)
texts = [raw_text]
metadatas = [metadata]
config = PreprocessorConfig()
print(preprocess_batch(texts, metadatas, config))

[PreprocessorResult(text='omg! angry angry check this out: link share with', metadata=Metadata(source='twitter', language='unknown', timestamp='2025-11-18T13:06:00Z', discarded=False, notes=['handles_removed', 'emoji_map', 'punct_normalized']))]


# Package Setup

Here's how to set up the package structure:

1.  Create a directory named `nlp`.
2.  Inside `nlp`, create a subdirectory named `src`.
3.  Inside `src`, create three files: `__init__.py`, `preprocess.py`, and `lang_detect.py`.
4.  Copy the content from the code blocks below into their respective files.

```python
# nlp/src/__init__.py
"""
Package initializer for the NLP source modules.

Exports:
- preprocess: Entry points and configuration for text cleaning.
- lang_detect: Lightweight language detection hook.
"""

from .preprocess import (
    Metadata,
    PreprocessorConfig,
    PreprocessorResult,
    preprocess,
    preprocess_batch,
)
from .lang_detect import detect_language
```

```python
# nlp/src/preprocess.py

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

# Local import for language detection
from .lang_detect import detect_language


@dataclass
class Metadata:
    """
    Metadata describing the text input and processing outcomes.

    Fields:
    - source: Optional identifier for the source platform (e.g., 'twitter', 'reddit').
    - language: ISO language code detected or provided (e.g., 'en'). 'unknown' if detection fails.
    - timestamp: Optional raw ingest timestamp as string or int (no transformation performed).
    - discarded: True if the input is considered non-processable (empty or only noise).
    - notes: Arbitrary processing notes (e.g., 'lang_unsupported', 'url_only').
    """
    source: Optional[str] = None
    language: str = "unknown"
    timestamp: Optional[Any] = None
    discarded: bool = False
    notes: List[str] = field(default_factory=list)


@dataclass
class PreprocessorResult:
    """
    Result structure holding cleaned text and updated metadata.

    Fields:
    - text: Cleaned text string or None if discarded.
    - metadata: Updated Metadata instance with detection outcomes and flags.
    """
    text: Optional[str]
    metadata: Metadata


@dataclass
class PreprocessorConfig:
    """
    Configurable options for the text preprocessing pipeline.

    Fields:
    - lower_case: Convert text to lowercase. Some languages may prefer preserving case.
    - emoji_policy: 'remove' to strip emojis, 'map' to convert to sentiment tokens.
    - remove_handles: Remove social media handles like '@username'.
    - remove_urls: Remove hyperlinks (http/https/www and bare domains).
    - remove_html_markdown: Strip HTML tags and basic markdown formatting.
    - normalize_punctuation: Collapse repeated punctuation (e.g., '!!!' -> '!').
    - preserve_year_like_numbers: Keep 4-digit numbers likely to be years; remove other excessive numbers if noisy.
    - remove_stopwords: Remove common stopwords while preserving negations.
    - supported_languages: List of ISO codes allowed; if empty, all languages are allowed.
    - route_unsupported_to_multilingual: If True, do not discard unsupported languages; leave metadata note.
    - min_content_chars: Minimum number of alphabetic characters required to keep the text (post-cleaning).
    """
    lower_case: bool = True
    emoji_policy: str = "map"  # 'map' or 'remove'
    remove_handles: bool = True
    remove_urls: bool = True
    remove_html_markdown: bool = True
    normalize_punctuation: bool = True
    preserve_year_like_numbers: bool = True
    remove_stopwords: bool = False
    supported_languages: List[str] = field(default_factory=list)  # e.g., ['en', 'fr']
    route_unsupported_to_multilingual: bool = True
    min_content_chars: int = 2


URL_RE = re.compile(
    r"""(?xi)
    \b
    (
        (https?://|www\.)[^\s]+
        |
        [a-z0-9.-]+\.[a-z]{2,}(/[^\s]*)?
    )
    """
)

HANDLE_RE = re.compile(r"(^|[\s\W])@[\w_]{1,50}", re.UNICODE)

HTML_TAG_RE = re.compile(r"<[^>]+>")
MD_FORMAT_RE = re.compile(r"(?m)(`{1,3}.*?`{1,3}|\*{1,2}|_{1,2}|~~)")

PUNCT_REPEAT_RE = re.compile(r"([!?.,;:])\1+")
WHITESPACE_RE = re.compile(r"\s+")

# Unicode emoji ranges (basic approximation); can be extended.
EMOJI_RE = re.compile(
    r"[\U0001F300-\U0001F5FF"  # Misc Symbols and Pictographs
    r"\U0001F600-\U0001F64F"   # Emoticons
    r"\U0001F680-\U0001F6FF"   # Transport & Map Symbols
    r"\U0001F700-\U0001F77F"   # Alchemical Symbols
    r"\U0001F780-\U0001F7FF"   # Geometric Shapes Extended
    r"\U0001F800-\U0001F8FF"   # Supplemental Arrows-C
    r"\U0001F900-\U0001F9FF"   # Supplemental Symbols and Pictographs
    r"\U0001FA00-\U0001FA6F"   # Symbols and Pictographs Extended-A
    r"\U00002700-\U000027BF"   # Dingbats
    r"\U00002600-\U000026FF]"  # Misc Symbols
, flags=re.UNICODE)

# Accept digits; we may remove non-year-like numbers.
NON_YEAR_NUMBER_RE = re.compile(r"\b(?!\d{4}\b)\d+\b")


# --- Emoji sentiment map (basic) ---
EMOJI_SENTIMENT_MAP = {
    "🙂": "smile",
    "😊": "smile",
    "😃": "smile",
    "😀": "smile",
    "🤣": "laugh",
    "😂": "laugh",
    "😉": "wink",
    "😍": "love",
    "😘": "kiss",
    "😡": "angry",
    "🤬": "angry",
    "😠": "angry",
    "😢": "sad",
    "😭": "cry",
    "😞": "sad",
    "😔": "sad",
    "😐": "neutral",
    "😕": "confused",
    "😬": "grimace",
    "😱": "shock",
    "😮": "surprised",
    "👍": "thumbs_up",
    "👎": "thumbs_down",
    "🔥": "fire",
    "💯": "hundred",
    "✨": "sparkle",
    "❤️": "heart",
    "💔": "broken_heart",
}


# --- Stopword sets with preserved negations ---
EN_STOPWORDS = {
    "the", "a", "an", "and", "or", "if", "then", "but", "so", "to", "of", "in",
    "on", "for", "with", "from", "by", "as", "at", "that", "this", "it", "is",
    "are", "was", "were", "be", "been", "being", "do", "does", "did", "can",
    "could", "should", "would", "may", "might", "will", "shall", "have", "has",
    "had", "about", "into", "over", "under", "up", "down", "out", "off"
    # Note: Negations intentionally excluded: 'not', 'never', 'no'
}

FR_STOPWORDS = {
    "le", "la", "les", "un", "une", "des", "et", "ou", "si", "alors", "mais",
    "donc", "de", "du", "dans", "sur", "pour", "avec", "par", "comme", "à",
    "que", "ce", "cela", "il", "elle", "est", "sont", "était", "étaient",
    "être", "été", "ayant", "fait", "faites", "peut", "pourrait", "devrait",
    "voudrait", "peut-être", "avoir", "a", "ont"
    # Negations excluded: 'ne', 'pas', 'jamais', 'rien'
}


def _strip_whitespace_and_case(text: str, lower_case: bool) -> str:
    """
    Normalize whitespace and case.

    Inputs:
    - text: Raw input text.
    - lower_case: If True, convert to lowercase.

    Output:
    - Normalized text with trimmed boundaries and optional lowercase.
    """
    text = text.strip()
    if lower_case:
        text = text.lower()
    # Replace various whitespace forms with a single space
    text = WHITESPACE_RE.sub(" ", text)
    return text


def _remove_urls(text: str) -> str:
    """
    Remove hyperlinks and bare domains.

    Inputs:
    - text: Current text.

    Output:
    - Text with URLs/domains removed.
    """
    return URL_RE.sub(" ", text)


def _remove_handles(text: str) -> str:
    """
    Remove social media handles (@username).

    Inputs:
    - text: Current text.

    Output:
    - Text with handles removed.
    """
    # Replace the handle with a single space while preserving preceding boundary
    return re.sub(r"(^|[\s\W])@[\w_]{1,50}", r"\1 ", text)


def _normalize_punctuation(text: str) -> str:
    """
    Collapse repeated punctuation and normalize spacing around punctuation.

    Inputs:
    - text: Current text.

    Output:
    - Text with normalized punctuation.
    """
    text = PUNCT_REPEAT_RE.sub(r"\1", text)
    # Remove stray non-word symbols except basic punctuation
    text = re.sub(r"[^\w\s.!?,;:'\"-]", " ", text)
    return text


def _handle_emojis(text: str, policy: str) -> Tuple[str, bool]:
    """
    Handle emojis per policy.

    Inputs:
    - text: Current text.
    - policy: 'remove' or 'map'.

    Output:
    - (Processed text, had_emoji)
    """
    had_emoji = False
    if policy == "remove":
        if EMOJI_RE.search(text):
            had_emoji = True
        text = EMOJI_RE.sub(" ", text)
        return text, had_emoji
    elif policy == "map":
        # Replace known emojis with sentiment tokens, strip unknowns
        def repl(match: re.Match) -> str:
            nonlocal had_emoji
            ch = match.group(0)
            had_emoji = True
            token = EMOJI_SENTIMENT_MAP.get(ch)
            return f" {token} " if token else " "

        text = EMOJI_RE.sub(repl, text)
        return text, had_emoji
    else:
        # Unknown policy; leave text unchanged
        return text, had_emoji


def _remove_noisy_numbers(text: str, preserve_year_like: bool) -> str:
    """
    Remove numbers considered noisy while preserving likely years (4-digit).

    Inputs:
    - text: Current text.
    - preserve_year_like: If True, preserve 4-digit numbers.

    Output:
    - Text with noisy numbers removed.
    """
    if preserve_year_like:
        text = NON_YEAR_NUMBER_RE.sub(" ", text)
    else:
        text = re.sub(r"\b\d+\b", " ", text)
    return text


def _remove_stopwords(text: str, language: str) -> str:
    """
    Optionally remove stopwords based on detected language, preserving negations.

    Inputs:
    - text: Current text.
    - language: ISO language code (e.g., 'en', 'fr').

    Output:
    - Text with stopwords removed for English or French; noop for others.
    """
    if language == "en":
        stopwords = EN_STOPWORDS
    elif language == "fr":
        stopwords = FR_STOPWORDS
    else:
        return text

    # Token-level removal with whitespace preservation
    tokens = text.split()
    filtered = [t for t in tokens if t not in stopwords]
    return " ".join(filtered)


def _final_spacing(text: str) -> str:
    """
    Ensure clean spacing and remove artifacts post-processing.

    Inputs:
    - text: Current text.

    Output:
    - Sanitized text with single spaces and trimmed ends.
    """
    text = WHITESPACE_RE.sub(" ", text)
    text = text.strip()
    return text


def _is_only_noise(text: str, min_content_chars: int) -> bool:
    """
    Determine if text is effectively noise (too few alphabetic characters).

    Inputs:
    - text: Current text.
    - min_content_chars: Minimum count of alphabetic characters required.

    Output:
    - True if considered noise, else False.
    """
    # Count alphabetic characters (any Unicode letter)
    alpha_chars = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]", text)
    return len(alpha_chars) < min_content_chars


def preprocess(
    text: Optional[str],
    metadata: Optional[Metadata] = None,
    config: Optional[PreprocessorConfig] = None,
) -> PreprocessorResult:
    """
    Preprocess a single raw text input.

    Inputs:
    - text: Raw text string (may be None or empty).
    - metadata: Optional Metadata object; if None, a new one is created.
    - config: Optional PreprocessorConfig; defaults applied if None.

    Outputs:
    - PreprocessorResult with cleaned text (or None if discarded) and updated metadata.

    Logic flow:
    1. Validate input; if empty, mark discarded.
    2. Normalize whitespace and case.
    3. Remove URLs and handles per config.
    4. Strip HTML and markdown artifacts if enabled.
    5. Handle emojis (remove or map).
    6. Normalize punctuation and remove noisy numbers.
    7. Detect language; decide stopword handling.
    8. Remove stopwords if enabled; keep negations.
    9. Final spacing and noise check.
    10. Route unsupported languages per configuration.
    """
    md = metadata or Metadata()
    cfg = config or PreprocessorConfig()

    if text is None:
        md.discarded = True
        md.notes.append("empty_input")
        return PreprocessorResult(text=None, metadata=md)

    # Step 1: Early trim and case
    current = _strip_whitespace_and_case(text, cfg.lower_case)

    # If after trimming we have nothing, discard
    if not current:
        md.discarded = True
        md.notes.append("empty_after_trim")
        return PreprocessorResult(text=None, metadata=md)

    # Step 2: Remove URLs and handles
    if cfg.remove_urls:
        before = current
        current = _remove_urls(current)
        if before != current:
            md.notes.append("urls_removed")

    if cfg.remove_handles:
        before = current
        current = _remove_handles(current)
        if before != current:
            md.notes.append("handles_removed")


    # Step 4: Emoji handling
    current, had_emoji = _handle_emojis(current, cfg.emoji_policy)
    if had_emoji:
        md.notes.append(f"emoji_{cfg.emoji_policy}")

    # Step 5: Punctuation normalization
    if cfg.normalize_punctuation:
        before = current
        current = _normalize_punctuation(current)
        if before != current:
            md.notes.append("punct_normalized")

    # Step 6: Number handling
    before = current
    current = _remove_noisy_numbers(current, cfg.preserve_year_like_numbers)
    if before != current:
            md.notes.append("numbers_processed")

    # Step 7: Language detection
    try:
        detected = detect_language(current) if current else "unknown"
    except Exception:
        detected = "unknown"
        md.notes.append("lang_detect_error")
    md.language = detected or "unknown"

    # Step 8: Stopword handling (optional)
    if cfg.remove_stopwords:
        before = current
        current = _remove_stopwords(current, md.language)
        if before != current:
            md.notes.append("stopwords_removed")

    # Step 9: Final spacing and noise check
    current = _final_spacing(current)
    if not current or _is_only_noise(current, cfg.min_content_chars):
        md.discarded = True
        md.notes.append("only_noise")
        return PreprocessorResult(text=None, metadata=md)

    # Step 10: Language routing / support check
    if cfg.supported_languages and md.language not in cfg.supported_languages:
        if cfg.route_unsupported_to_multilingual:
            md.notes.append("lang_unsupported_routed")
        else:
            md.discarded = True
            md.notes.append("lang_unsupported_discarded")
            return PreprocessorResult(text=None, metadata=md)

    return PreprocessorResult(text=current, metadata=md)


def preprocess_batch(
    texts: List[Optional[str]],
    metadatas: Optional[List[Optional[Metadata]]] = None,
    config: Optional[PreprocessorConfig] = None,
) -> List[PreprocessorResult]:
    """
    Preprocess a batch of texts efficiently.

    Inputs:
    - texts: List of raw text strings (items may be None).
    - metadatas: Optional list of Metadata; aligns by index. Missing entries create default Metadata.
    - config: Optional PreprocessorConfig applied to all items.

    Outputs:
    - List of PreprocessorResult in the same order as inputs.

    Notes:
    - Designed for large batches; processing steps are lightweight and avoid per-item heavy allocations.
    """
    cfg = config or PreprocessorConfig()
    results: List[PreprocessorResult] = []

    md_list = metadatas or [None] * len(texts)
    if len(md_list) != len(texts):
        raise ValueError("Length of metadatas must match length of texts.")

    for i, raw in enumerate(texts):
        md = md_list[i] if isinstance(md_list[i], Metadata) else None
        results.append(preprocess(raw, md, cfg))

    return results
```

```python
# nlp/src/lang_detect.py
# This is a stub for language detection. In a real-world scenario, you would integrate a robust language detection library here.

# nlp/src/lang_detect.py
"""
Lightweight language detection hook for preprocessing.

Purpose:
- Provide a simple detect_language(text) function returning ISO-like codes ('en', 'fr', etc.).
- Serve as an interface placeholder until replaced by a production-ready detector.

Heuristic:
- Count stopword matches for supported languages (English, French) and pick the dominant.
- Fallback to 'unknown' if signals are weak or text too short.

Notes:
- This is intentionally simple to keep dependencies minimal.
- Replace the implementation with a proper detector (e.g., fastText, langid) as needed.
"""

import re
from typing import Optional

# Basic stopword signals; should align with preprocessing stopwords to avoid conflicts.
EN_SIG = {
    "the", "and", "or", "to", "of", "in", "on", "for", "with", "is", "it", "this", "that"
}
FR_SIG = {
    "le", "la", "les", "et", "ou", "de", "du", "dans", "sur", "pour", "avec", "est", "ce", "cela"
}

MIN_LEN = 10  # minimum characters to attempt detection


def detect_language(text: Optional[str]) -> str:
    """
    Detect language using a lightweight heuristic.

    Inputs:
    - text: Cleaned or raw text.

    Outputs:
    - ISO-like language code ('en', 'fr', 'unknown').

    Logic:
    - If text is too short or None, return 'unknown'.
    - Score English/French by presence of common function words; choose the higher score.
      If tie or both zero, return 'unknown'.
    """
    if not text:
        return "unknown"

    normalized = text.lower()
    stripped = re.sub(r"[^\w\sÀ-ÖØ-öø-ÿ]", " ", normalized)
    tokens = stripped.split()
    if len(stripped) < MIN_LEN or not tokens:
        return "unknown"

    en_score = sum(1 for t in tokens if t in EN_SIG)
    fr_score = sum(1 for t in tokens if t in FR_SIG)

    if en_score > fr_score and en_score >= 2:
        return "en"
    if fr_score > en_score and fr_score >= 2:
        return "fr"
    return "unknown"


# Sentiment Analysis

In [ ]:
pip install torch transformers

In [25]:
# nlp/src/model.py
"""
Module: model.py
Scope:
    Provides a reusable interface for loading a pre-trained transformer model and tokenizer
    for emotion classification (or other NLP tasks) with robust device handling and
    environment-driven configuration.

Key Features:
    - Loads model and tokenizer by name (defaulting to a practical multilingual model).
    - Detects GPU vs CPU automatically (cuda if available, otherwise cpu).
    - Supports batch size configuration via environment variables.
    - Exposes a single public function get_model() that returns (model, tokenizer, device, config).
    - Caches loaded artifacts to avoid repeated initialization across the pipeline.
    - Graceful error handling with informative exceptions and logging hints.

Environment Variables:
    - MODEL_NAME: Hugging Face model identifier (e.g., "roberta-base", "bert-base-uncased", "xlm-roberta-base").
                  Default: "xlm-roberta-base" for multilingual coverage.
    - BATCH_SIZE: Integer batch size for inference/training pipeline. Default: 16.

Usage:
    from nlp.src.model import get_model

    model, tokenizer, device, config = get_model()
    # Use model and tokenizer, move tensors to `device`, and honor config['batch_size'] in batching.
"""

from __future__ import annotations

import os
import logging
from typing import Any, Dict, Tuple, Optional

# Torch and Transformers are required dependencies.
try:
    import torch
except ImportError as e:
    raise ImportError(
        "PyTorch is required but not installed. Please install with `pip install torch` "
        "or follow instructions at https://pytorch.org/get-started/locally/."
    ) from e

try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        PreTrainedTokenizerBase,
        PreTrainedModel,
    )
except ImportError as e:
    raise ImportError(
        "Transformers is required but not installed. Please install with `pip install transformers`."
    ) from e

# Configure a module-level logger for optional integration with downstream logging.
logger = logging.getLogger(__name__)
logger.addHandler(logging.NullHandler())

# Module-level cache to avoid reloading across calls.
_CACHED: Dict[str, Any] = {
    "model": None,
    "tokenizer": None,
    "device": None,
    "config": None,
}

# Sensible defaults: multilingual model for broader coverage; moderate batch size.
DEFAULT_MODEL_NAME = "xlm-roberta-base"
DEFAULT_BATCH_SIZE = 16


class ModelLoadError(RuntimeError):
    """Raised when the transformer model or tokenizer cannot be loaded properly."""


def _detect_device() -> torch.device:
    """
    Detects the available compute device.

    Returns:
        torch.device: 'cuda' if available, otherwise 'cpu'.

    Notes:
        - Prefers CUDA if PyTorch reports availability.
        - Could be extended for MPS (Apple Silicon) or other accelerators in future.
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        logger.info("CUDA available. Using GPU.")
        return device
    # Consider Apple Silicon (MPS) if desired; here we default to CPU for simplicity.
    device = torch.device("cpu")
    logger.info("CUDA not available. Using CPU.")
    return device


def _get_env_config() -> Dict[str, Any]:
    """
    Reads environment variables for model configuration.

    Returns:
        Dict[str, Any]: Configuration dictionary containing:
            - 'model_name': Hugging Face model name.
            - 'batch_size': Integer batch size for inference/training.

    Error Handling:
        - Falls back to defaults if vars are missing or invalid.
        - Logs warnings for invalid values.
    """
    model_name = os.getenv("MODEL_NAME", DEFAULT_MODEL_NAME)

    batch_size_raw = os.getenv("BATCH_SIZE", str(DEFAULT_BATCH_SIZE))
    try:
        batch_size = int(batch_size_raw)
        if batch_size <= 0:
            raise ValueError("batch size must be positive")
    except Exception as exc:
        logger.warning(
            "Invalid BATCH_SIZE=%r. Using default=%d. Reason: %s",
            batch_size_raw,
            DEFAULT_BATCH_SIZE,
            exc,
        )
        batch_size = DEFAULT_BATCH_SIZE

    return {
        "model_name": model_name,
        "batch_size": batch_size,
    }


def _load_tokenizer(model_name: str) -> PreTrainedTokenizerBase:
    """
    Loads the tokenizer corresponding to the given model.

    Args:
        model_name (str): Hugging Face model identifier.

    Returns:
        PreTrainedTokenizerBase: Loaded tokenizer.

    Raises:
        ModelLoadError: If tokenizer cannot be loaded.
    """
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        return tokenizer
    except Exception as exc:
        msg = (
            f"Failed to load tokenizer for model '{model_name}'. "
            "Ensure the model name is correct and that you have network access or a local cache."
        )
        logger.error("%s Exception: %s", msg, exc)
        raise ModelLoadError(msg) from exc


def _load_model(model_name: str) -> PreTrainedModel:
    """
    Loads a sequence classification model for the given identifier.

    Args:
        model_name (str): Hugging Face model identifier.

    Returns:
        PreTrainedModel: Loaded model instance suitable for sequence classification.

    Raises:
        ModelLoadError: If model cannot be loaded or is incompatible.
    """
    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        return model
    except Exception as exc:
        msg = (
            f"Failed to load model '{model_name}'. This may indicate the identifier is incorrect, "
            "the model is not available for sequence classification, or there are connectivity issues."
        )
        logger.error("%s Exception: %s", msg, exc)
        raise ModelLoadError(msg) from exc


def get_model(
    force_reload: bool = False,
) -> Tuple[PreTrainedModel, PreTrainedTokenizerBase, torch.device, Dict[str, Any]]:
    """
    Public API: Returns a ready-to-use model, tokenizer, device, and config.

    Args:
        force_reload (bool): If True, ignore cached artifacts and reload from source.
                             Useful if environment variables changed at runtime.

    Returns:
        Tuple[PreTrainedModel, PreTrainedTokenizerBase, torch.device, Dict[str, Any]]:
            - model: Hugging Face PreTrainedModel for sequence classification.
            - tokenizer: Matching tokenizer for the model.
            - device: torch.device ('cuda' or 'cpu'), already set on model.
            - config: dict with 'model_name' and 'batch_size'.

    Behavior:
        - Lazy-caches loaded model and tokenizer to avoid repeated downloads/initialization.
        - Moves model to the selected device automatically.

    Raises:
        ModelLoadError: If loading fails for model or tokenizer.
        ImportError: If required libraries are missing.
    """
    # Return cached artifacts unless reload requested.
    if not force_reload and _CACHED["model"] is not None:
        return _CACHED["model"], _CACHED["tokenizer"], _CACHED["device"], _CACHED["config"]

    # Read env-driven configuration.
    config = _get_env_config()
    model_name = config["model_name"]

    # Detect device (prefer CUDA).
    device = _detect_device()

    # Load tokenizer first for early failure visibility.
    tokenizer = _load_tokenizer(model_name)

    # Load model; we opt for a sequence classification head.
    model = _load_model(model_name)

    # Move model to the selected device.
    try:
        model.to(device)
    except Exception as exc:
        msg = f"Loaded model could not be moved to device {device}. Check device and model compatibility."
        logger.error("%s Exception: %s", msg, exc)
        raise ModelLoadError(msg) from exc

    # Save to cache for subsequent calls.
    _CACHED["model"] = model
    _CACHED["tokenizer"] = tokenizer
    _CACHED["device"] = device
    _CACHED["config"] = config

    logger.info(
        "Model '%s' loaded successfully on device=%s with batch_size=%d",
        model_name,
        device,
        config["batch_size"],
    )

    return model, tokenizer, device, config


Classifyer

In [28]:
# nlp/src/classify.py
"""
Module: classify.py
Scope:
    Provides batch emotion classification for cleaned comments using a transformer model.

Features:
    - Accepts list of strings (cleaned comments).
    - Tokenizes, pads, and truncates inputs to max length.
    - Runs inference in batches for efficiency.
    - Returns probability distributions across 8 emotions, dominant emotion, confidence score.
    - Skips empty/invalid inputs gracefully with metadata flag.
    - Supports GPU acceleration automatically.
    - Batch size configurable via environment variables (from model.py).

Dependencies:
    - Requires model.py for get_model() to load model, tokenizer, device, and config.
    - Requires torch and transformers libraries.

Usage:
    from nlp.src.classify import classify_batch

    results = classify_batch(["I feel happy today!", "This is terrible..."])
    for r in results:
        print(r)
"""

from __future__ import annotations

import logging
from typing import List, Dict, Any

import torch
import torch.nn.functional as F

logger = logging.getLogger(__name__)
logger.addHandler(logging.NullHandler())

# Define the emotion labels expected from the model.
# Adjust if the model has a different label mapping.
EMOTION_LABELS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise",
    "trust",
]


def classify_batch(comments: List[str]) -> List[Dict[str, Any]]:
    """
    Classifies a batch of cleaned comments into emotion categories.

    Args:
        comments (List[str]): List of cleaned comment strings.

    Returns:
        List[Dict[str, Any]]: For each comment, returns a dictionary with:
            - 'input': Original input string.
            - 'probabilities': List of floats (probability distribution across emotions).
            - 'dominant_emotion': String label of highest probability emotion.
            - 'confidence': Float probability of dominant emotion.
            - 'discarded': Bool flag if input was empty/invalid and skipped.
    """
    model, tokenizer, device, config = get_model()
    batch_size = config.get("batch_size", 16)

    results: List[Dict[str, Any]] = []

    # Filter and prepare inputs
    valid_indices = []
    valid_comments = []
    for idx, comment in enumerate(comments):
        if not isinstance(comment, str) or not comment.strip():
            results.append(
                {
                    "input": comment,
                    "probabilities": [],
                    "dominant_emotion": None,
                    "confidence": None,
                    "discarded": True,
                }
            )
        else:
            valid_indices.append(idx)
            valid_comments.append(comment.strip())

    if not valid_comments:
        return results

    # Tokenize in batches
    encodings = tokenizer(
        valid_comments,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    model.eval()
    with torch.no_grad():
        for start in range(0, len(valid_comments), batch_size):
            end = start + batch_size
            batch_input_ids = input_ids[start:end]
            batch_attention_mask = attention_mask[start:end]

            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
            )

            # Assume model outputs logits for classification
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)

            # Process each item in batch
            for i, prob_tensor in enumerate(probs):
                prob_list = prob_tensor.cpu().numpy().tolist()
                max_idx = int(torch.argmax(prob_tensor).cpu().item())
                dominant_emotion = (
                    EMOTION_LABELS[max_idx]
                    if max_idx < len(EMOTION_LABELS)
                    else str(max_idx)
                )
                confidence = prob_list[max_idx]

                results.append(
                    {
                        "input": valid_comments[start + i],
                        "probabilities": prob_list,
                        "dominant_emotion": dominant_emotion,
                        "confidence": confidence,
                        "discarded": False,
                    }
                )

    return results


In [ ]:
# nlp/src/types.py
"""
Module: types.py
Scope:
    Defines consistent data structures for emotion classification outputs across the NLP pipeline.

Features:
    - Provides type-safe schema for classification results.
    - Ensures clarity and consistency when passing results between modules.
    - Supports metadata including language code and moderation/discard flags.
    - Defines batch result container for multiple comments.

Usage:
    from nlp.src.types import ClassificationResult, BatchResult

    result = ClassificationResult(
        text="I feel happy today!",
        language="en",
        probabilities={"joy": 0.85, "sadness": 0.05, "anger": 0.02, "fear": 0.01,
                       "disgust": 0.01, "surprise": 0.03, "trust": 0.02, "neutral": 0.01},
        dominant_emotion="joy",
        confidence=0.85,
        discarded=False,
        moderated=False,
    )

    batch = BatchResult(results=[result])
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List


# Define the canonical set of emotion labels expected in classification.
EMOTION_LABELS: List[str] = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise",
    "trust",
]


@dataclass
class ClassificationResult:
    """
    Represents the classification output for a single comment.

    Attributes:
        text (str): Original cleaned text input.
        language (str): Language code (e.g., 'en', 'fr').
        probabilities (Dict[str, float]): Mapping of emotion → probability (0–1).
        dominant_emotion (str): Emotion label with highest probability.
        confidence (float): Probability of the dominant emotion.
        discarded (bool): Flag indicating if input was invalid/empty and skipped.
        moderated (bool): Flag indicating if input was moderated/filtered.
    """
    text: str
    language: str
    probabilities: Dict[str, float] = field(default_factory=dict)
    dominant_emotion: str = ""
    confidence: float = 0.0
    discarded: bool = False
    moderated: bool = False

    def validate(self) -> None:
        """
        Validates the classification result for consistency.

        Raises:
            ValueError: If probabilities do not sum to ~1.0 or dominant_emotion not in EMOTION_LABELS.
        """
        if self.probabilities:
            total = sum(self.probabilities.values())
            if not (0.99 <= total <= 1.01):
                raise ValueError(
                    f"Probabilities must sum to 1.0 (±0.01). Got {total:.4f}."
                )
        if self.dominant_emotion and self.dominant_emotion not in EMOTION_LABELS:
            raise ValueError(
                f"Dominant emotion '{self.dominant_emotion}' not in allowed labels {EMOTION_LABELS}."
            )


@dataclass
class BatchResult:
    """
    Represents the classification output for a batch of comments.

    Attributes:
        results (List[ClassificationResult]): List of classification results.
    """
    results: List[ClassificationResult] = field(default_factory=list)

    def add_result(self, result: ClassificationResult) -> None:
        """
        Adds a classification result to the batch.

        Args:
            result (ClassificationResult): Result to add.
        """
        self.results.append(result)

    def discarded_count(self) -> int:
        """
        Returns:
            int: Number of discarded results in the batch.
        """
        return sum(1 for r in self.results if r.discarded)

    def moderated_count(self) -> int:
        """
        Returns:
            int: Number of moderated results in the batch.
        """
        return sum(1 for r in self.results if r.moderated)
